# RoPE vs Positional Embedding in BERT

Transformers have no built-in sense of token order — self-attention is a **set operation**.  
Positional encodings inject order information into the model.

## The Problem

Given tokens `[A, B, C]` the attention score between A and B is:

```
score(A, B) = (A · Wq) · (B · Wk)ᵀ
```

This is **position-blind**: swapping B and C produces identical scores.  
We need to inject the *relative* or *absolute* position of each token.

## The Three Approaches

| Method | Where position is injected | Extrapolates? | Used in |
|---|---|---|---|
| Sinusoidal (Vaswani 2017) | Added to token embedding | Somewhat | Original Transformer |
| Learned Absolute (BERT) | Learned embedding table, added | No | BERT, GPT-2 |
| RoPE (Su 2021) | Multiplied into Q/K inside attention | Yes | LLaMA, Mistral, Gemma |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

plt.rcParams.update({'figure.facecolor': 'white', 'axes.grid': True,
                     'grid.alpha': 0.3, 'font.size': 11})

torch.manual_seed(42)

D_MODEL   = 64    # embedding dim (small for clarity)
MAX_SEQ   = 128   # max sequence length
N_HEADS   = 4
HEAD_DIM  = D_MODEL // N_HEADS   # 16

print(f'd_model={D_MODEL}, n_heads={N_HEADS}, head_dim={HEAD_DIM}')

---
## Part 1 — Sinusoidal Positional Encoding (Original Transformer)

Vaswani et al. (2017) proposed **fixed** sinusoidal functions:

$$PE_{(pos, 2i)}   = \sin\left(\frac{pos}{10000^{2i/d}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

- Even dimensions use `sin`, odd dimensions use `cos`
- Each dimension oscillates at a different frequency
- Low dimensions: high frequency (changes every token)
- High dimensions: low frequency (changes slowly — captures global position)

The key property: $PE_{pos+k}$ can be expressed as a **linear function** of $PE_{pos}$,  
so the model can learn to attend to relative positions.

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    """
    Fixed sinusoidal positional encoding (Vaswani et al. 2017).
    Creates a (1, max_seq_len, d_model) buffer — no learned parameters.
    """
    def __init__(self, d_model: int, max_seq_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # PE matrix: (max_seq_len, d_model)
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len).unsqueeze(1).float()  # (T, 1)

        # div_term: 1 / 10000^(2i/d_model)  for i = 0, 1, ..., d_model/2
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model)
        )  # (d_model/2,)

        pe[:, 0::2] = torch.sin(position * div_term)   # even dims
        pe[:, 1::2] = torch.cos(position * div_term)   # odd dims

        # Register as buffer (moves with .to(device), not a parameter)
        self.register_buffer('pe', pe.unsqueeze(0))    # (1, T, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model)
        return self.dropout(x + self.pe[:, :x.size(1)])


# ── Visualise ────────────────────────────────────────────────────────
sinpe = SinusoidalPositionalEncoding(D_MODEL, MAX_SEQ, dropout=0.0)
pe_matrix = sinpe.pe.squeeze(0).numpy()   # (MAX_SEQ, D_MODEL)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

im = axes[0].imshow(pe_matrix[:50, :], aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
plt.colorbar(im, ax=axes[0])
axes[0].set_title('Sinusoidal PE — first 50 positions × 64 dims')
axes[0].set_xlabel('Dimension'); axes[0].set_ylabel('Position')

# Show a few individual dimensions to illustrate frequency differences
for dim, color in [(0, 'crimson'), (4, 'steelblue'), (16, 'darkgreen'), (60, 'orange')]:
    axes[1].plot(pe_matrix[:80, dim], label=f'dim {dim}', color=color, linewidth=1.8)
axes[1].set_title('PE values for individual dimensions')
axes[1].set_xlabel('Position'); axes[1].set_ylabel('PE value')
axes[1].legend()

plt.tight_layout(); plt.show()
print('Low dimensions (0, 4): high frequency — distinguish nearby positions')
print('High dimensions (16, 60): low frequency — encode global position')

---
## Part 2 — BERT Learned Absolute Positional Embedding

BERT (Devlin et al. 2019) replaces sinusoidal encoding with a **learned embedding table**:

```
token_embedding     = TokenEmbedding(input_ids)       # (B, T, d)
position_embedding  = PositionEmbedding(positions)    # (B, T, d)  ← learned table
segment_embedding   = SegmentEmbedding(segment_ids)   # (B, T, d)

x = LayerNorm(token_embedding + position_embedding + segment_embedding)
```

The position embedding is a standard `nn.Embedding(max_position_embeddings, hidden_size)`.  
BERT uses `max_position_embeddings = 512`.

### Key Limitation: No Extrapolation

Because positions 0–511 are learned independently, the model has **never seen position 512**.  
Feeding a sequence longer than 512 tokens produces undefined behaviour.

In [ ]:
class BERTPositionalEmbedding(nn.Module):
    """
    BERT-style learned absolute positional embedding.
    Creates a trainable lookup table of shape (max_seq_len, d_model).
    """
    def __init__(self, d_model: int, max_seq_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)
        self.dropout = nn.Dropout(dropout)
        # Standard BERT init: N(0, 0.02)
        nn.init.normal_(self.pos_embedding.weight, mean=0.0, std=0.02)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model)
        B, T, _ = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0)  # (1, T)
        return self.dropout(x + self.pos_embedding(positions))       # (B, T, d_model)


class BERTEmbeddingLayer(nn.Module):
    """Full BERT embedding: token + position + segment."""
    def __init__(self, vocab_size: int, d_model: int, max_seq_len: int = 512,
                 n_segments: int = 2, dropout: float = 0.1):
        super().__init__()
        self.token     = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.position  = nn.Embedding(max_seq_len, d_model)
        self.segment   = nn.Embedding(n_segments, d_model)
        self.norm      = nn.LayerNorm(d_model)
        self.dropout   = nn.Dropout(dropout)

        nn.init.normal_(self.token.weight,    std=0.02)
        nn.init.normal_(self.position.weight, std=0.02)
        nn.init.normal_(self.segment.weight,  std=0.02)

    def forward(self, token_ids: torch.Tensor, segment_ids: torch.Tensor | None = None):
        B, T = token_ids.shape
        positions   = torch.arange(T, device=token_ids.device).unsqueeze(0)
        if segment_ids is None:
            segment_ids = torch.zeros_like(token_ids)

        x = self.token(token_ids) + self.position(positions) + self.segment(segment_ids)
        return self.dropout(self.norm(x))


# ── Test ─────────────────────────────────────────────────────────────
bert_emb = BERTEmbeddingLayer(vocab_size=30522, d_model=D_MODEL, max_seq_len=MAX_SEQ)
token_ids   = torch.randint(1, 30522, (2, 16))    # (batch=2, seq=16)
segment_ids = torch.zeros(2, 16, dtype=torch.long)
out = bert_emb(token_ids, segment_ids)
print(f'BERT embedding output shape: {out.shape}')   # (2, 16, 64)

# Visualise the learned position table (after random init)
pe_weights = bert_emb.position.weight.detach().numpy()  # (MAX_SEQ, D_MODEL)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
im = axes[0].imshow(pe_weights[:50, :], aspect='auto', cmap='viridis')
plt.colorbar(im, ax=axes[0])
axes[0].set_title('BERT Position Embedding weights (random init, not yet trained)')
axes[0].set_xlabel('Dimension'); axes[0].set_ylabel('Position')

# Cosine similarity between position embeddings
norms = np.linalg.norm(pe_weights[:32], axis=1, keepdims=True)
normed = pe_weights[:32] / (norms + 1e-9)
sim = normed @ normed.T
im2 = axes[1].imshow(sim, cmap='RdYlGn', vmin=-1, vmax=1)
plt.colorbar(im2, ax=axes[1])
axes[1].set_title('Cosine similarity between position embeddings (random init)')
axes[1].set_xlabel('Position j'); axes[1].set_ylabel('Position i')

plt.tight_layout(); plt.show()
print('After training, nearby positions tend to have higher cosine similarity.')

---
## Part 3 — RoPE (Rotary Position Embedding)

**Su et al. (2021)** — used in LLaMA, Mistral, PaLM, Gemma, Qwen.

### Core Idea

Instead of adding a position vector to the embedding, RoPE **rotates** the query and key vectors  
by an angle proportional to the token's position **before** the dot-product attention.

The rotation is designed so that the dot product `q · k` naturally encodes **relative position**:

$$\text{score}(q_m, k_n) = (R_m q) \cdot (R_n k) = q^\top R_m^\top R_n k = q^\top R_{m-n} k$$

The score depends only on $(m - n)$ — the **relative** distance, not the absolute positions.

### The Rotation Matrix

For a 2D pair of dimensions $(x_{2i}, x_{2i+1})$ at position $m$:

$$\begin{pmatrix} x'_{2i} \\ x'_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix}$$

where $\theta_i = 10000^{-2i/d}$ (same base as sinusoidal PE).

### Efficient Implementation (Complex Number Form)

The rotation can be computed without explicit matrix multiplication:  
treat each pair $(x_{2i}, x_{2i+1})$ as a complex number $z = x_{2i} + j \cdot x_{2i+1}$,  
and multiply by $e^{jm\theta_i} = \cos(m\theta_i) + j\sin(m\theta_i)$.

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    """
    Rotary Position Embedding (RoPE) — Su et al. 2021.

    Applied to query and key tensors inside attention, NOT to the
    token embeddings directly (unlike BERT/sinusoidal PE).

    Usage:
        rope = RotaryPositionalEmbedding(head_dim)
        q_rot = rope(q, seq_len)   # (B, n_heads, T, head_dim)
        k_rot = rope(k, seq_len)
        scores = q_rot @ k_rot.transpose(-2, -1) / sqrt(head_dim)
    """

    def __init__(self, head_dim: int, base: float = 10000.0):
        super().__init__()
        # θᵢ = 1 / base^(2i/d)  for i in 0..d//2
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer('inv_freq', inv_freq)   # (head_dim/2,)
        self.head_dim = head_dim

    def _build_cos_sin(self, seq_len: int, device: torch.device):
        """
        Build cos and sin tables for positions 0 .. seq_len-1.

        Returns: cos, sin  each of shape (seq_len, head_dim)
        """
        t = torch.arange(seq_len, device=device).float()          # (T,)
        # Outer product: t[m] * inv_freq[i]  → (T, head_dim/2)
        freqs = torch.outer(t, self.inv_freq)                      # (T, head_dim/2)
        # Repeat so we cover all head_dim dims (pair each dim with its partner)
        emb = torch.cat([freqs, freqs], dim=-1)                   # (T, head_dim)
        return emb.cos(), emb.sin()                                # (T, head_dim) each

    @staticmethod
    def _rotate_half(x: torch.Tensor) -> torch.Tensor:
        """
        Rotate the second half of the head dim to implement the
        2D rotation without explicit matrix multiplication.

        For pair (x₀, x₁): rotate_half gives (-x₁, x₀)
        Then: x*cos + rotate_half(x)*sin
              = (x₀·cos - x₁·sin,  x₁·cos + x₀·sin)   ← 2D rotation ✓
        """
        x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, x: torch.Tensor, seq_len: int | None = None) -> torch.Tensor:
        """
        Apply RoPE rotation to x.

        x: (batch, n_heads, seq_len, head_dim)
        Returns rotated tensor of same shape.
        """
        if seq_len is None:
            seq_len = x.shape[-2]
        cos, sin = self._build_cos_sin(seq_len, x.device)  # (T, head_dim)
        # Broadcast over batch and heads: (1, 1, T, head_dim)
        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)
        return x * cos + self._rotate_half(x) * sin


# ── Verify rotation preserves vector norm ────────────────────────────
rope = RotaryPositionalEmbedding(HEAD_DIM)
q = torch.randn(1, N_HEADS, 8, HEAD_DIM)   # (B, H, T, d_head)
q_rot = rope(q)

norms_before = q.norm(dim=-1)
norms_after  = q_rot.norm(dim=-1)
print(f'Norm before rotation: {norms_before[0,0,:4].tolist()}')
print(f'Norm after  rotation: {norms_after[0,0,:4].tolist()}')
print(f'Max norm change: {(norms_after - norms_before).abs().max().item():.2e}')
print('✓ RoPE is an isometry — it preserves the L2 norm of every vector.')

---
## Part 4 — RoPE Encodes Relative Position in Attention Scores

The fundamental property of RoPE: the dot product between rotated q and k  
depends only on their **relative** offset `m - n`, not on absolute positions `m` and `n`.

In [ ]:
def attention_score_vs_offset(q_vec, k_vec, rope, max_offset=32):
    """
    Compute dot(RoPE(q, pos=m), RoPE(k, pos=n)) as a function of (m - n).
    With RoPE, this should depend only on the offset, not on absolute position.
    """
    scores_by_offset = {}

    for offset in range(-max_offset, max_offset + 1):
        # Try several absolute positions to confirm it only depends on offset
        scores = []
        for base_pos in [0, 10, 50, 100]:
            m = base_pos
            n = base_pos - offset     # m - n = offset
            if n < 0:
                continue
            T = max(m, n) + 1
            # Expand to (1, 1, T, head_dim) and apply RoPE
            q_full = q_vec.unsqueeze(0).unsqueeze(0).unsqueeze(0).expand(1, 1, T, -1)
            k_full = k_vec.unsqueeze(0).unsqueeze(0).unsqueeze(0).expand(1, 1, T, -1)
            q_rot = rope(q_full)
            k_rot = rope(k_full)
            score = (q_rot[0, 0, m] * k_rot[0, 0, n]).sum().item()
            scores.append(score)
        if scores:
            scores_by_offset[offset] = scores

    return scores_by_offset


torch.manual_seed(7)
q_vec = torch.randn(HEAD_DIM)
k_vec = torch.randn(HEAD_DIM)

scores_rope = attention_score_vs_offset(q_vec, k_vec, rope, max_offset=24)

offsets = sorted(scores_rope.keys())
mean_scores  = [np.mean(scores_rope[o]) for o in offsets]
std_scores   = [np.std(scores_rope[o])  for o in offsets]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(offsets, mean_scores, color='crimson', linewidth=2)
axes[0].fill_between(offsets,
    [m - s for m, s in zip(mean_scores, std_scores)],
    [m + s for m, s in zip(mean_scores, std_scores)],
    alpha=0.2, color='crimson')
axes[0].set_title('RoPE: Attention score vs relative offset (m-n)')
axes[0].set_xlabel('Relative offset (m - n)')
axes[0].set_ylabel('q_rotated · k_rotated')
axes[0].axvline(0, color='gray', linestyle='--', linewidth=1)

# Show std across absolute positions → near zero → score depends only on offset
axes[1].plot(offsets, std_scores, color='steelblue', linewidth=2)
axes[1].set_title('Std across different absolute positions (should ≈ 0)')
axes[1].set_xlabel('Relative offset'); axes[1].set_ylabel('Std')
axes[1].set_yscale('log')

plt.tight_layout(); plt.show()
print(f'Max std across absolute positions: {max(std_scores):.2e}')
print('→ Score depends ONLY on relative offset — absolute positions cancel out.')

---
## Part 5 — Full Attention Heads: BERT-style vs RoPE

Putting it all together — implement one attention head with each approach.

In [ ]:
import math

class BERTSelfAttention(nn.Module):
    """
    Scaled dot-product self-attention with BERT absolute position embeddings.
    Position info lives in the token representation (added before attention).
    """
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads
        self.scale    = math.sqrt(self.head_dim)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None):
        # x: (B, T, d_model)  — position already embedded
        B, T, _ = x.shape
        H, D = self.n_heads, self.head_dim

        Q = self.q_proj(x).view(B, T, H, D).transpose(1, 2)  # (B, H, T, D)
        K = self.k_proj(x).view(B, T, H, D).transpose(1, 2)
        V = self.v_proj(x).view(B, T, H, D).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / self.scale  # (B, H, T, T)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        out = (attn_weights @ V).transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(out), attn_weights


class RoPESelfAttention(nn.Module):
    """
    Scaled dot-product self-attention with RoPE applied to Q and K.
    Position info is injected INSIDE attention via rotation — NOT in embeddings.
    """
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads
        self.scale    = math.sqrt(self.head_dim)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

        # RoPE applied per head
        self.rope = RotaryPositionalEmbedding(self.head_dim)

    def forward(self, x: torch.Tensor, mask: torch.Tensor | None = None):
        # x: (B, T, d_model)  — NO position embedding added beforehand
        B, T, _ = x.shape
        H, D = self.n_heads, self.head_dim

        Q = self.q_proj(x).view(B, T, H, D).transpose(1, 2)  # (B, H, T, D)
        K = self.k_proj(x).view(B, T, H, D).transpose(1, 2)
        V = self.v_proj(x).view(B, T, H, D).transpose(1, 2)

        # ← KEY DIFFERENCE: rotate Q and K with their positions before dot-product
        Q = self.rope(Q)
        K = self.rope(K)

        scores = (Q @ K.transpose(-2, -1)) / self.scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        out = (attn_weights @ V).transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(out), attn_weights


# ── Test both ─────────────────────────────────────────────────────────
B, T = 2, 16

# For BERT: position is added to the input embeddings
bert_pos = BERTPositionalEmbedding(D_MODEL, MAX_SEQ, dropout=0.0)
x_raw    = torch.randn(B, T, D_MODEL)
x_bert   = bert_pos(x_raw)     # position embedded in x

bert_attn = BERTSelfAttention(D_MODEL, N_HEADS)
rope_attn = RoPESelfAttention(D_MODEL, N_HEADS)

out_bert, w_bert = bert_attn(x_bert)
out_rope, w_rope = rope_attn(x_raw)   # raw input — RoPE handles position inside

print(f'BERT attention output : {out_bert.shape}  weights: {w_bert.shape}')
print(f'RoPE attention output : {out_rope.shape}  weights: {w_rope.shape}')

# Visualise attention patterns
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, w, title in [
    (axes[0], w_bert[0, 0].detach(), 'BERT Self-Attention (head 0)'),
    (axes[1], w_rope[0, 0].detach(), 'RoPE Self-Attention (head 0)'),
]:
    im = ax.imshow(w.numpy(), cmap='Blues', vmin=0, vmax=w.max().item())
    plt.colorbar(im, ax=ax)
    ax.set_title(title); ax.set_xlabel('Key position'); ax.set_ylabel('Query position')
plt.suptitle('Attention weights (random init — not trained)', fontsize=12)
plt.tight_layout(); plt.show()

---
## Part 6 — Why RoPE Extrapolates and BERT Does Not

In [ ]:
# BERT: Attempting a sequence longer than max_seq_len crashes or gives garbage
# RoPE: Works for any sequence length — just computes more cos/sin values

LONG_SEQ = 256  # > MAX_SEQ used during "training"

# RoPE handles it gracefully
x_long = torch.randn(1, LONG_SEQ, D_MODEL)
out_long, w_long = rope_attn(x_long)
print(f'RoPE with seq_len={LONG_SEQ}: output shape {out_long.shape}  ✓')

# Visualise how RoPE attention score decays with distance (locality bias)
# This emerges naturally from the rotation — distant positions rotate to be orthogonal
torch.manual_seed(0)
q_single = torch.randn(1, 1, 1, HEAD_DIM)  # (B, H, 1, D) — one query

# Place the key at positions 0 .. 63 and measure dot product with query at pos 0
rope_eval = RotaryPositionalEmbedding(HEAD_DIM)
k_vec_fixed = torch.randn(HEAD_DIM)
scores_at_offsets = []

MAX_EVAL_POS = 64
for n in range(MAX_EVAL_POS):
    T = n + 1
    q_t = q_single.expand(1, 1, T, HEAD_DIM)
    k_t = k_vec_fixed.view(1, 1, 1, HEAD_DIM).expand(1, 1, T, HEAD_DIM)
    q_r = rope_eval(q_t)
    k_r = rope_eval(k_t)
    # score: q at pos 0 vs k at pos n
    s = (q_r[0, 0, 0] * k_r[0, 0, n]).sum().item()
    scores_at_offsets.append(s)

plt.figure(figsize=(10, 4))
plt.plot(scores_at_offsets, color='crimson', linewidth=2)
plt.title('RoPE: Score between query@pos=0 and key@pos=n (same key vector)')
plt.xlabel('Key position n'); plt.ylabel('Attention score')
plt.axhline(0, color='gray', linestyle='--', linewidth=1)
plt.tight_layout(); plt.show()

print('RoPE naturally encodes a recency bias:')
print('Nearby tokens (small offset) → higher/more varied scores')
print('Distant tokens (large offset) → scores oscillate toward zero (rotation causes cancellation)')

---
## Part 7 — Head-to-Head Comparison

In [ ]:
print('=' * 70)
print(f'{"Property":<30} {"BERT PE":<20} {"RoPE"}')
print('=' * 70)
rows = [
    ('Type',              'Learned absolute',   'Fixed rotation (no params)'),
    ('Where applied',     'Added to embedding', 'Rotated inside Q, K'),
    ('Encodes',           'Absolute position',  'Relative position'),
    ('Extrapolation',     'No (hard limit 512)', 'Yes (any length)'),
    ('Trainable params',  'max_len × d_model',  '0'),
    ('Norm preserved',    'No',                 'Yes (isometry)'),
    ('Works with KV cache','No — pos in input', 'Yes — pos applied at decoding'),
    ('Models',            'BERT, GPT-2',        'LLaMA, Mistral, Gemma, Qwen'),
]
for prop, bert, rope in rows:
    print(f'{prop:<30} {bert:<20} {rope}')
print('=' * 70)

# Parameter count comparison
bert_params = MAX_SEQ * D_MODEL
rope_params = 0
print(f'\nBERT PE parameters (128 × 64) : {bert_params:,}')
print(f'RoPE parameters               : {rope_params}')
print(f'\n(For BERT-base: 512 × 768 = {512*768:,} position params)')

---
## Summary

### BERT Absolute Positional Embedding
- Adds a **learned vector** for each absolute position to the token embedding
- Simple, effective for fixed-length tasks (classification, NER, QA ≤ 512 tokens)
- **Cannot generalise** beyond the maximum trained sequence length
- Position information flows through Q, K, V projections

### RoPE
- **Rotates** Q and K vectors inside attention by an angle proportional to position
- The dot product `qᵀk` automatically captures **relative** position (m − n)
- **Zero extra parameters** — rotation angles are fixed by a formula
- **Extrapolates** naturally to unseen lengths (though at reduced quality beyond ≈ 4× training length — addressed by YaRN, LongRoPE)
- Required for modern autoregressive models with **KV cache** (each token's position is applied at decode time, not at embedding time)

**For new projects**: Use RoPE unless you specifically need BERT-style bidirectional encoding with a fixed context length.